# EDA — Indian Weather (T024)

Notebook de **análise exploratória** para o marco M02 / história S02. Restrição da equipa: **PyArrow** + **Matplotlib** + **NumPy** (sem pandas nem scikit-learn).

## Amostra e reprodutibilidade

A primeira célula de código define `N_AMOSTRA` (linhas lidas do Parquet após seleção de colunas). **Gráficos e estatísticas descritas aqui referem-se a essa amostra**, não necessariamente ao ficheiro completo, salvo indicação em contrário.

## Documentação relacionada

- Alvo e pipeline: [preprocessamento.md](../docs/preprocessamento.md)
- Desequilíbrio de classes: [imbalance.md](../docs/imbalance.md)


In [7]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Tamanho da amostra (linhas) após leitura com colunas selecionadas
N_AMOSTRA = 500_000


def resolve_repo_and_fig_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    roots = [cwd]
    if cwd.name == "notebooks":
        roots.append(cwd.parent)
    for root in roots:
        if (root / "requirements.txt").exists():
            fig = root / "notebooks" / "figuras"
            fig.mkdir(parents=True, exist_ok=True)
            return root, fig
    fig = cwd / "figuras"
    fig.mkdir(parents=True, exist_ok=True)
    return cwd, fig


REPO_ROOT, FIG_DIR = resolve_repo_and_fig_dir()
PARQUET = REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet"

os.environ.setdefault("MPLBACKEND", "Agg")

COLS_LEITURA = [
    "datetime",
    "rain_label",
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
    "lat",
    "lon",
    "hour",
    "month",
]

if not PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {PARQUET}. Coloque o dataset em data/ e execute a partir da raiz do repositório ou abra o Jupyter com cwd na raiz."
    )

table = pq.read_table(PARQUET, columns=COLS_LEITURA, use_threads=True)
n_full = table.num_rows
if table.num_rows > N_AMOSTRA:
    table = table.slice(0, N_AMOSTRA)

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", FIG_DIR)
print("Linhas no ficheiro (aprox.):", n_full)
print("Linhas usadas neste notebook:", table.num_rows)
print(table.schema)


REPO_ROOT: C:\Desenvolvimento\BigData
FIG_DIR: C:\Desenvolvimento\BigData\notebooks\figuras
Linhas no ficheiro (aprox.): 46082160
Linhas usadas neste notebook: 500000
datetime: timestamp[ms]
rain_label: int64
temperature_C: double
humidity_pct: int64
precip_mm: double
cloud_cover_pct: int64
pressure_hPa: double
dew_point_C: double
wind_speed_ms: double
solar_radiation_Wm2: double
lat: double
lon: double
hour: int64
month: int64


## Distribuição do alvo (`rain_label`)

Histograma de frequências absolutas. Para interpretação do desequilíbrio e estratégias (class weights, oversampling no Spark, etc.), ver [imbalance.md](../docs/imbalance.md).


In [8]:
labels = table.column("rain_label")
vc = pc.value_counts(labels)
labs = vc.field(0).to_numpy(zero_copy_only=False)
cnts = vc.field(1).to_numpy(zero_copy_only=False)
total = float(cnts.sum())

fig, ax = plt.subplots(figsize=(6, 4))
xs = [str(x) for x in labs]
bars = ax.bar(xs, cnts, color="#4C72B0", edgecolor="#222222", linewidth=0.8)
ax.set_xlabel("rain_label")
ax.set_ylabel("Contagem (amostra)")
ax.set_title("Distribuição do alvo na amostra")
for rect, n in zip(bars, cnts):
    pct = 100.0 * float(n) / total
    ax.annotate(
        f"{int(n):,}\n({pct:.1f}%)",
        xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax.margins(x=0.15)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_rain_label_distribuicao.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_rain_label_distribuicao.png")
for a, b in zip(labs, cnts):
    print(a, int(b))


Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_rain_label_distribuicao.png
0 459562
1 40438


## Relações feature–alvo

Médias de variáveis numéricas por classe (`rain_label`) via `Table.group_by` do PyArrow, e **boxplot** de `precip_mm` por classe (distribuição condicional na amostra).


In [9]:
agg_cols = [
    ("temperature_C", "mean", "°C"),
    ("humidity_pct", "mean", "%"),
    ("cloud_cover_pct", "mean", "%"),
]
g = table.group_by("rain_label").aggregate([(c, a) for c, a, _ in agg_cols])
print(g)

labs_mean = g.column("rain_label").to_pylist()
x = np.arange(len(labs_mean))
width = 0.55

fig, axes = plt.subplots(3, 1, figsize=(7, 7.2), sharex=True)
for ax, (col, _, unit) in zip(axes, agg_cols):
    vals = g.column(f"{col}_mean").to_numpy(zero_copy_only=False)
    bars = ax.bar(x, vals, width, color="#4C72B0", edgecolor="#222222", linewidth=0.6)
    ax.set_ylabel(f"média ({unit})")
    ax.set_title(f"{col} por rain_label")
    for rect, v in zip(bars, vals):
        ax.annotate(
            f"{v:.2f}",
            xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )
axes[-1].set_xticks(x)
axes[-1].set_xticklabels([str(v) for v in labs_mean])
axes[-1].set_xlabel("rain_label")
fig.suptitle("Médias por classe — uma escala por variável (amostra)", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_medias_num_por_rain_label.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_medias_num_por_rain_label.png")


pyarrow.Table
rain_label: int64
temperature_C_mean: double
humidity_pct_mean: double
cloud_cover_pct_mean: double
----
rain_label: [[0,1]]
temperature_C_mean: [[26.251443983619236,25.66298531084626]]
humidity_pct_mean: [[65.09294066959409,89.1622978386666]]
cloud_cover_pct_mean: [[45.21409733615921,93.4283841930857]]
Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_medias_num_por_rain_label.png


In [10]:
y = table.column("rain_label").to_numpy(zero_copy_only=False)
p = table.column("precip_mm").to_numpy(zero_copy_only=False)
classes = np.unique(y[~np.isnan(y)])
data = []
labels_txt = []
for c in classes:
    m = y == c
    vals = p[m & ~np.isnan(p)]
    data.append(vals)
    labels_txt.append(str(int(c)) if float(c).is_integer() else str(c))

fig, ax = plt.subplots(figsize=(7, 4.6))
bp = ax.boxplot(data, showfliers=False, patch_artist=True)
colors = ("#8da0cb", "#fc8d62")
for patch, color in zip(bp["boxes"], colors[: len(bp["boxes"])]):
    patch.set_facecolor(color)
    patch.set_edgecolor("#222222")
    patch.set_linewidth(0.9)
for whisker in bp["whiskers"]:
    whisker.set(color="#333333", linewidth=0.9)
for cap in bp["caps"]:
    cap.set(color="#333333", linewidth=0.9)
for median in bp["medians"]:
    median.set(color="#111111", linewidth=1.4)
ax.set_xticks(np.arange(1, len(labels_txt) + 1))
ax.set_xticklabels(labels_txt)
ax.set_xlabel("rain_label")
ax.set_ylabel("precip_mm")
ax.set_title("precip_mm por classe (amostra; outliers omitidos)")
ax.text(
    0.02,
    0.98,
    "Esperado: rain_label=0 com precipitação nula → caixa colapsada em 0.",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="wheat", alpha=0.88),
)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_precip_boxplot_por_rain_label.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_precip_boxplot_por_rain_label.png")


Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_precip_boxplot_por_rain_label.png


## Correlação entre preditores (amostra)

Subconjunto de colunas numéricas alinhadas a [preprocessamento.md](../docs/preprocessamento.md). A matriz usa `numpy.corrcoef`; valores ausentes são substituídos temporariamente pela média da coluna **só para este cálculo exploratório** (não substitui decisões do pipeline T022).


In [11]:
corr_cols = [
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
]
rows = []
for c in corr_cols:
    v = table.column(c).to_numpy(zero_copy_only=False).astype("float64", copy=False)
    rows.append(v)
X = np.vstack(rows)
col_means = np.nanmean(X, axis=1, keepdims=True)
X_filled = np.where(np.isnan(X), col_means, X)
C = np.corrcoef(X_filled)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Correlação de Pearson — amostra; NaN imputados pela média da coluna")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_correlacao_preditoras.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_correlacao_preditoras.png")


Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_correlacao_preditoras.png


## Dimensão temporal (`hour`, `month`, `datetime`)

Exploração de sazonalidade horária e mensal e de contagem de registos por dia (data truncada a partir de `datetime`).

As contagens por dia usam **apenas as primeiras `N_AMOSTRA` linhas** do Parquet, na ordem em que o ficheiro foi lido; se essa fatia for temporalmente contígua e com cadência fixa por dia, o gráfico pode aparecer em **patamares** — isso reflecte a estrutura da amostra, não um erro de ordenação (os dias são ordenados antes do plot).


In [12]:
h_raw = table.column("hour").to_numpy(zero_copy_only=False)
m_raw = table.column("month").to_numpy(zero_copy_only=False)
h = h_raw[~np.isnan(h_raw)].astype(np.int64, copy=False)
m = m_raw[~np.isnan(m_raw)].astype(np.int64, copy=False)
h = np.clip(h, 0, 23)
m = np.clip(m, 1, 12)
c_h = np.bincount(h, minlength=24)
c_m = np.bincount(m - 1, minlength=12)
hours_x = np.arange(24)
months_x = np.arange(1, 13)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(hours_x, c_h, color="#55A868", edgecolor="#1a3d22", linewidth=0.65)
ax1.set_xlabel("hour (0–23)")
ax1.set_ylabel("Contagem")
ax1.set_title("Contagem por hora (amostra)")
ax1.set_xticks(np.arange(0, 24, 2))
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(months_x, c_m, color="#C44E52", edgecolor="#4a1518", linewidth=0.65)
ax2.set_xlabel("month (1–12)")
ax2.set_ylabel("Contagem")
ax2.set_title("Contagem por mês (amostra)")
ax2.set_xticks(months_x)
ax2.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_hour_month_histogramas.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_hour_month_histogramas.png")


Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_hour_month_histogramas.png


In [13]:
d_arr = pc.cast(table.column("datetime"), pa.date32())
st = pc.value_counts(d_arr)
days = st.field(0).to_numpy(zero_copy_only=False)
cnts = st.field(1).to_numpy(zero_copy_only=False)
order = np.argsort(days)
days_s = days[order]
cnts_s = cnts[order].astype(float)
x = np.asarray(days_s, dtype="datetime64[D]")

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.step(x, cnts_s, where="post", color="#4C72B0", linewidth=1.1, label="contagem/dia")
ax.scatter(x, cnts_s, s=10, color="#2b4c7e", alpha=0.55, zorder=3)
ax.set_xlabel("Data")
ax.set_ylabel("Nº de linhas por dia (só na amostra)")
ax.set_title("Contagem por dia — fatia inicial de N linhas do Parquet")
ax.grid(True, axis="y", alpha=0.35)
ax.legend(loc="upper right", fontsize=8)
fig.autofmt_xdate()
fig.text(
    0.5,
    0.01,
    "Patamares = cadência estável por dia nesta fatia; dias ordenados antes do gráfico.",
    ha="center",
    fontsize=8,
    transform=fig.transFigure,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG_DIR / "eda_contagem_por_dia.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_contagem_por_dia.png")


Figura: C:\Desenvolvimento\BigData\notebooks\figuras\eda_contagem_por_dia.png


## Figuras exportadas (caminhos)

Relativos à raiz do repositório:

- `notebooks/figuras/eda_rain_label_distribuicao.png`
- `notebooks/figuras/eda_medias_num_por_rain_label.png`
- `notebooks/figuras/eda_precip_boxplot_por_rain_label.png`
- `notebooks/figuras/eda_correlacao_preditoras.png`
- `notebooks/figuras/eda_hour_month_histogramas.png`
- `notebooks/figuras/eda_contagem_por_dia.png`

## Insights acionáveis (Parte 3)

1. **Alvo:** validar na amostra (e depois no Parquet completo com `profile_parquet` / Spark) se a distribuição de `rain_label` exige **class weights** ou **rebalanceamento** conforme [imbalance.md](../docs/imbalance.md); alinhar métricas (F1, PR-AUC) ao desequilíbrio.
2. **Chuva (`precip_mm`):** caudas pesadas por classe sugerem que modelos baseados em **árvores** ou transformações robustas podem ser mais estáveis do que assumir Gaussianidade; manter imputação/mediana **fit só em treino** (T022).
3. **Correlações fortes** (ex.: termodinâmica entre temperatura, humidade, ponto de orvalho): considerar **redundância** e custo de features em Spark; árvores lidam bem, modelos lineares podem precisar de regularização ou seleção.
4. **Temporal:** picos em certas horas/meses reforçam utilidade de `hour` / `month` (ou representação cíclica sin/cos) já prevista no pré-processamento; **split temporal** (T021) evita vazamento quando se exploram tendências por dia.
5. **Próximo passo:** materializar splits e pipeline em Spark com **fit apenas no treino** e validar generalização na janela temporal de validação/teste.
